# Make geomedian gifs

* If you want to change the size of the gif, edit the fig_size var in the generate_image_for_year function (line 13). For the geomedian story map gif I used (7,1). This function also sets where the text will sit (line 17). That will need to be changed for story map (I think it was 0.9, 0.85)

## Import stuff

In [1]:
import datacube
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import os

import glob
import imageio
import numpy as np
from PIL import Image

from odc.ui import with_ui_cbk
from datacube.utils.cog import write_cog

import sys
sys.path.insert(1, "../Tools/")
from dea_tools.plotting import rgb, display_map

## Set up functions

In [3]:
def make_gif(frame_folder, start_year, end_year, location):
    """
    Create a gif from a folder of images (png). 
    Parameters:
    - frame folder (str): path to the folder containing the images. 
    - start_year (int): starting year for the GIF title.
    - end_year (int): ending year for the GIF title.
    - location (str): location for the GIF title.
    Returns:
    - the gif will be saved in the current folder. 
    """
    frames = [Image.open(image) for image in sorted(glob.glob(f"{frame_folder}/*.png"))]
    frame_one = frames[0]
    frame_one.save(f"Geomedian {start_year}-{end_year} {location}.gif", format="GIF", append_images=frames,
               save_all=True, duration=500, loop=0, optimize=True)

In [4]:
def load_geomedian_data(product, measurements, lon_range, lat_range, year):
    """
    Load geomedian dataset for a given year.

    Parameters:
    - product (str): product name.
    - measurements (list): list of measurement names.
    - lon_range (tuple): longitude range.
    - lat_range (tuple): latitude range.
    - year (int): year for which data is to be loaded.

    Returns:
    - geomedian (xarray.Dataset): Loaded geomedian dataset.
    """
    dc = datacube.Datacube(app="")
    geomedian = dc.load(
        product=product,
        measurements=measurements,
        x=lon_range,
        y=lat_range,
        time=(str(year), str(year)),
    )
    return geomedian

In [5]:
def generate_image_for_year(geomedian_slice, year, location):
    """
    Generate an image for a specific year.

    Parameters:
    - geomedian_slice (xarray.Dataset): slice of geomedian dataset for the year.
    - year (int): year for which the image is generated.
    - location (str): location used in the image filename.

    Returns:
    - None
    """
    f, axarr = plt.subplots(1, 1, squeeze=False, layout="constrained") #, figsize=(5, 3))
    rgb(geomedian_slice, bands=["nbart_red", "nbart_green", "nbart_blue"], ax=axarr[0, 0])

    year_text = f'{year}'
    axarr[0, 0].text(0.95, 0.95, year_text, transform=axarr[0, 0].transAxes, fontsize=8,
                     ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))

    axarr[0, 0].set_title('')
    axarr[0, 0].set_xlabel('')
    axarr[0, 0].set_ylabel('')
    axarr[0, 0].set_xticks([])
    axarr[0, 0].set_yticks([])
    plt.axis('off')

    plt.savefig(f'images/{year}_{location}.png', dpi=150)
    plt.close()

In [6]:
def get_product_for_year(year):
    """
    Get the product name based on the given year.

    Parameters:
    - year (int): year for which product is needed.

    Returns:
    - product (str): product name.
    """
    if 1988 <= year <= 1999:
        return "ga_ls5t_gm_cyear_3"
    elif 2000 <= year <= 2003:
        return "ga_ls7e_gm_cyear_3"
    elif 2004 <= year <= 2007:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2008:
        return "ga_ls7e_gm_cyear_3"
    elif 2009 <= year <= 2011:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2012:
        return "ga_ls7e_gm_cyear_3"
    elif year >= 2013:
        return "ga_ls8cls9c_gm_cyear_3"
    else:
        return None

In [28]:
def delete_generated_images(folder):
    """
    Delete all the images from the 'images' directory.

    Parameters:
    - folder (str): name of folder/folder path

    Returns:
    - None
    """
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")

In [24]:
def generate_gif(start_year, end_year, lon_range, lat_range, location):
    """
    Generate images for a range of years.

    Parameters:
    - start_year (int): starting year.
    - end_year (int): ending year.
    - lon_range (tuple): longitude range.
    - lat_range (tuple): latitude range.
    - location (str): location used in the image filenames.

    Returns:
    - None
    """
    measurements = ["nbart_red", "nbart_green", "nbart_blue"]
    for year in range(start_year, end_year + 1):
        product = get_product_for_year(year)

        if product is None:
            print(f"Skipping year {year} as it doesn't match the specified ranges.")
            continue

        geomedian = load_geomedian_data(product, measurements, lon_range, lat_range, year)

        for i in range(len(geomedian.time)):
            geomedian_slice = geomedian.isel(time=i)
            current_time = geomedian_slice.time.values
            year = np.datetime_as_string(current_time, unit='Y').astype(int)
            generate_image_for_year(geomedian_slice, year, location)

    make_gif("images", start_year, end_year, location)

    delete_generated_images('images')

    print("Finished ( ͡° ͜ʖ ͡° )")

## Set up location

In [25]:
# location = "Shepparton, Vic"
# lat = -36.3657
# lon =  145.0623
# lat_buffer = 0.2
# lon_buffer = 0.3

# coords for long gif for story map
# location = "Shepparton, Vic"
# lat =-36.3824
# lon = 145.2600
# lat_buffer = 0.03
# lon_buffer = 0.3

# location = "Derby, WA"
# lat =  -17.2615
# lon = 123.5358
# lat_buffer = 0.
# lon_buffer = 0.

# location = "Canberra, ACT"
# lat = -35.3090
# lon =  149.1243
# lat_buffer = 0.18
# lon_buffer = 0.27

location = "Bauxite Mine, WA"
lat = -32.5549
lon = 116.1667
lat_buffer = 0.2
lon_buffer = 0.3

# location = "Shoemaker crater, WA"
# lat =  -25.7034
# lon = 120.4459
# lat_buffer = 0.52
# lon_buffer = 0.78

# location = "Lake Victoria, Vic"
# lat = -34.0663
# lon = 141.2842
# lat_buffer = 0.4
# lon_buffer = 0.6

# location = "Lake Dalrymple, Qld"
# lat =  -20.6636
# lon = 146.9971
# lat_buffer = 0.2
# lon_buffer = 0.3

# location = "Atherton, Qld"
# lat = -17.2303
# lon = 145.5218
# lat_buffer = 0.52
# lon_buffer = 0.78

# location = "Weipa, Qld"
# lat = -12.5841
# lon = 141.8541
# lat_buffer = 0.2
# lon_buffer = 0.3

# location = "Lake Argyle, WA"
# lat =  -16.3702
# lon =  128.7515
# lat_buffer = 0.4
# lon_buffer = 0.6

# location = "North Brisbane, Qld"
# lat = -27.2731
# lon = 152.9970
# lat_buffer = 0.1
# lon_buffer = 0.15


#=====================================================
lat_range = (lat - lat_buffer, lat + lat_buffer)
lon_range = (lon - lon_buffer, lon + lon_buffer)

In [26]:
display_map(x=lon_range, y=lat_range)

## Make gif

In [27]:
start_year = 2022
end_year = 2023
generate_gif(2020, 2023, lon_range, lat_range, location)

Finished ( ͡° ͜ʖ ͡° )
